In [7]:
data = torch.tensor(range(10))
data = data.reshape(2, 5)
data

tensor([[0, 1, 2, 3, 4],
        [5, 6, 7, 8, 9]])

In [10]:
data.size()

torch.Size([2, 5])

In [20]:
b_tensor = data.as_strided(data.permute(1,0).size(), (1,2))
b_tensor


tensor([[0, 2],
        [1, 3],
        [2, 4],
        [3, 5],
        [4, 6]])

In [21]:
ll = [0, 50, 100]

for i,j in zip(ll, ll[1:]):
    print(i,j)

0 50
50 100


In [22]:
data = torch.tensor(range(10))
offsets = [0, 5, 10]

groups = [
    data.as_strided((j - i,), (1,), i)
    for i, j in zip(offsets, offsets[1:])
]

In [23]:
groups

[tensor([0, 1, 2, 3, 4]), tensor([5, 6, 7, 8, 9])]

In [2]:
import torch
torch.__version__

'2.6.0+cu124'

In [3]:

import torch.nn as nn

class TinyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(10, 10, bias=False)
        self.layer2 = nn.Linear(10, 10, bias=False)

    def forward(self, x):
        return self.layer2(self.layer1(x))
    
model = TinyModel()
model


TinyModel(
  (layer1): Linear(in_features=10, out_features=10, bias=False)
  (layer2): Linear(in_features=10, out_features=10, bias=False)
)

In [5]:
for name, module in model.named_children():
    print(f"子模块名称: {name}, 类型: {type(module)}")

子模块名称: layer1, 类型: <class 'torch.nn.modules.linear.Linear'>
子模块名称: layer2, 类型: <class 'torch.nn.modules.linear.Linear'>


In [6]:
import torch.nn.functional as F
class TransformerBlock(nn.Module):
    def __init__(self, d_model=10, nhead=2, dim_feedforward=5, dropout=0.1):
        super().__init__()
        # 自注意力层
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        
        # 前馈神经网络
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        
        # 归一化层
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        # Dropout
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, mask=None):
        # 自注意力
        attn_output, _ = self.self_attn(x, x, x, attn_mask=mask)
        x = x + self.dropout(attn_output)
        x = self.norm1(x)
        
        # 前馈网络
        ff_output = self.linear2(F.gelu(self.linear1(x)))
        x = x + self.dropout(ff_output)
        x = self.norm2(x)
        
        return x
    
class TransformerModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleDict({  # 显式定义的模块容器
            f"layer_{i}": TransformerBlock() for i in range(2)
        })

transformer = TransformerModel()


In [ ]:
transformer


TransformerModel(
  (layers): ModuleDict(
    (layer_0): TransformerBlock(
      (self_attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=10, out_features=10, bias=True)
      )
      (linear1): Linear(in_features=10, out_features=5, bias=True)
      (linear2): Linear(in_features=5, out_features=10, bias=True)
      (norm1): LayerNorm((10,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((10,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (layer_1): TransformerBlock(
      (self_attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=10, out_features=10, bias=True)
      )
      (linear1): Linear(in_features=10, out_features=5, bias=True)
      (linear2): Linear(in_features=5, out_features=10, bias=True)
      (norm1): LayerNorm((10,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((10,), eps=1e-05, elementwise_affine=True)
      (dropou

In [12]:
import torch
from torch.nn import ModuleDict, Linear
import torch.nn as nn

class TinyDictModel(nn.Module):
    def __init__(self):
        super().__init__()
        # 用字典保存 4 个 Linear 层
        self.tiny_block = nn.ModuleDict({
            "linear_1": nn.Linear(10, 10),
            "linear_2": nn.Linear(10, 10),
        })

    def forward(self, x):
        for lid, block in self.layers.items():
            x = block(x)
        return x

t_model = TinyDictModel()
t_model

TinyDictModel(
  (tiny_block): ModuleDict(
    (linear_1): Linear(in_features=10, out_features=10, bias=True)
    (linear_2): Linear(in_features=10, out_features=10, bias=True)
  )
)

In [10]:
for name, param in t_model.named_parameters():
    print(f"{name:<30} shape={tuple(param.shape)}  dtype={param.dtype}  device={param.device}\n [Params]:\n{param}")

layers.linear_1.weight         shape=(10, 10)  dtype=torch.float32  device=cpu
 [Params]:
Parameter containing:
tensor([[-0.1369, -0.0387, -0.2796, -0.2026, -0.2704,  0.2404,  0.2222,  0.2200,
         -0.2660, -0.2430],
        [ 0.0243, -0.1939,  0.1837,  0.1603,  0.2817, -0.0039,  0.0950,  0.2696,
          0.2656,  0.2657],
        [-0.1485, -0.2240, -0.0818, -0.2328,  0.2861, -0.2993,  0.0039, -0.1849,
          0.0934, -0.1540],
        [ 0.1216,  0.1104, -0.0487, -0.0711,  0.1369,  0.1285, -0.2570, -0.3029,
          0.2930,  0.0966],
        [ 0.2732, -0.0955, -0.2867, -0.2778, -0.3123, -0.1530,  0.2095,  0.0963,
          0.1374,  0.0273],
        [ 0.1382, -0.1723, -0.0323, -0.1953, -0.2268, -0.1649, -0.0138, -0.0588,
          0.0821,  0.0343],
        [ 0.1421,  0.1267,  0.2159,  0.0749, -0.2466,  0.2088, -0.3036, -0.0513,
         -0.2065,  0.0522],
        [-0.2166, -0.0128,  0.1788, -0.0046,  0.2013,  0.1005,  0.2621,  0.1394,
         -0.1574,  0.2621],
        [ 0.1870